# コイン集めアリーナ — クラス対戦演習

これまでの発展教材（クラス → 簡単なゲーム → バイナリとセキュリティ → ネットワークプログラミング）の **総まとめ** です。

**教室のみんなのPython botが、1つの盤面で同時に競います。**

## ゲームのルール

- 盤面（24×24マス）に **コイン** がいくつか出現します
- 各プレイヤー（＝あなたの **bot**）は、上下左右に1マスずつ動けます
- コインの上に乗ると **+1点**。コインは別の場所に再出現します
- 制限時間で、**得点が一番高い人の勝ち**

## 2つの役割

- **先生**: サーバ（審判＋盤面）を立て、**公開URL** を配り、盤面をスクリーンに映す
- **生徒**: サーバに接続する **bot** を書く。良い戦略を考えた人が勝つ

使う技術は全部これまでの復習です：**HTTPクライアント/サーバ・JSON・bot**（ネットワーク回）、最後の発展で **ハッシュ／署名**（バイナリ回）も出てきます。

---
# 共通: アリーナのコード

次の **2つのセル**（盤面の見た目 と サーバ本体）は、**先生**、および **ソロ練習する生徒** が実行します。アリーナを動かすための土台です。
（本番で先生のURLに接続するだけなら、実行は不要です。）

まず下のセルで、盤面の見た目（**JavaScript**）を用意します。折りたたまれています。中身のJS（盤面の見た目）に興味があれば、左の ▶ を押すと展開できます。

In [ ]:
#@title ▶ 盤面の見た目 { display-mode: "form" }
# 盤面のHTML/JSを board.html に書き出す（サーバがこれを配信する）
board_html = r'''
<!doctype html><html><head><meta charset="utf-8"><title>コイン集めアリーナ</title>
<style>
 body{background:#0d1117;color:#e6edf3;font-family:sans-serif;text-align:center;margin:0}
 canvas{background:#161b22;border-radius:8px;margin-top:10px}
 #rank{display:inline-block;text-align:left;margin:8px auto;font-size:15px}
</style></head>
<body>
 <h2>コイン集めアリーナ</h2>
 <canvas id="c" width="480" height="480"></canvas>
 <div id="rank"></div>
 <script>
   const cell = 20;
   const ctx = document.getElementById('c').getContext('2d');
   async function tick() {
     let s;
     try { s = await (await fetch('/state')).json(); } catch (e) { return; }
     ctx.clearRect(0, 0, 480, 480);
     // コイン（黄色）
     ctx.fillStyle = '#f1c40f';
     for (const c of s.coins) {
       ctx.beginPath();
       ctx.arc(c[0] * cell + cell / 2, c[1] * cell + cell / 2, 6, 0, 7);
       ctx.fill();
     }
     // プレイヤー（各自の色）
     for (const p of s.players) {
       ctx.fillStyle = p.color;
       ctx.beginPath();
       ctx.arc(p.x * cell + cell / 2, p.y * cell + cell / 2, 8, 0, 7);
       ctx.fill();
       ctx.fillStyle = '#fff';
       ctx.font = '10px sans-serif';
       ctx.fillText(p.name, p.x * cell - 4, p.y * cell - 4);
     }
     // ランキング
     const r = [...s.players].sort((a, b) => b.score - a.score);
     document.getElementById('rank').innerHTML = '<b>ランキング</b><br>' +
       r.map((p, i) => `${i + 1}. <span style="color:${p.color}">■</span> ${p.name}: ${p.score}`).join('<br>');
   }
   setInterval(tick, 150);
 </script>
</body></html>
'''
with open("board.html", "w", encoding="utf-8") as f:
    f.write(board_html)
print("board.html を用意しました。")

次がサーバ本体です。**ネットワーク回で学んだ `http.server` の応用**なので、興味があれば読んでみてください。

In [ ]:
# === アリーナサーバ（ネットワーク回で学んだ http.server の応用）===
# ※ 先に「board.html を書き出すセル」を実行しておくこと
import json, threading, time, random
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

W, H, NUM_COINS = 24, 24, 6          # 盤面の広さ と コインの数
_lock = threading.Lock()
players = {}                          # id -> {name,x,y,score,color}
coins = []
_next_id = [0]
def _rand_pos(): return [random.randint(0, W-1), random.randint(0, H-1)]
for _ in range(NUM_COINS): coins.append(_rand_pos())

class Arena(BaseHTTPRequestHandler):
    def _send(self, obj, code=200):
        b = json.dumps(obj).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Access-Control-Allow-Origin", "*")   # どこからでも接続可
        self.end_headers(); self.wfile.write(b)
    def _body(self):
        n = int(self.headers.get("Content-Length", 0))
        return json.loads(self.rfile.read(n) or b"{}")
    def do_GET(self):
        if self.path == "/" or self.path.startswith("/?"):
            with open("board.html", encoding="utf-8") as f:   # 盤面HTMLを毎回読み込んで返す
                body = f.read().encode()
            self.send_response(200)
            self.send_header("Content-Type", "text/html; charset=utf-8")
            self.end_headers(); self.wfile.write(body)
        elif self.path == "/state":
            with _lock:
                self._send({"w": W, "h": H, "coins": coins,
                    "players": [{"id": i, "name": p["name"], "x": p["x"], "y": p["y"],
                                 "score": p["score"], "color": p["color"]} for i, p in players.items()]})
        else:
            self._send({"error": "not found"}, 404)
    def do_POST(self):
        d = self._body()
        if self.path == "/join":
            with _lock:
                i = _next_id[0]; _next_id[0] += 1; pos = _rand_pos()
                players[i] = {"name": str(d.get("name", "noname"))[:12], "x": pos[0], "y": pos[1],
                              "score": 0, "color": "#%06x" % random.randint(0x333333, 0xffffff)}
                self._send({"id": i, "w": W, "h": H})
        elif self.path == "/move":
            with _lock:
                p = players.get(d.get("id"))
                if not p: self._send({"error": "unknown id"}, 400); return
                dx, dy = {"up": (0,-1), "down": (0,1), "left": (-1,0), "right": (1,0)}.get(d.get("dir"), (0,0))
                p["x"] = max(0, min(W-1, p["x"] + dx)); p["y"] = max(0, min(H-1, p["y"] + dy))
                for c in coins:                       # 乗ったコインを回収
                    if c[0] == p["x"] and c[1] == p["y"]:
                        p["score"] += 1; c[0], c[1] = _rand_pos()
                self._send({"x": p["x"], "y": p["y"], "score": p["score"]})
        else:
            self._send({"error": "not found"}, 404)
    def log_message(self, *a): pass

def start_server():
    """アリーナサーバを別スレッドで起動し、(server, port) を返す"""
    srv = ThreadingHTTPServer(("0.0.0.0", 0), Arena)
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    time.sleep(0.3)
    return srv, srv.server_address[1]

print("アリーナの準備OK。start_server() で起動できます。")

---
# A. 先生セクション

先生だけが実行します。サーバを起動し、公開URLを発行し、盤面をスクリーンに映します。

In [ ]:
# 【先生】サーバを起動する
server, PORT = start_server()
print("サーバ起動 ポート:", PORT)

## 公開URLを発行する

`cloudflared` で、教室の全員が接続できる **公開URL**（`https://xxxx.trycloudflare.com`）を発行します。

In [ ]:
#@title ▶ 公開URLを発行（cloudflared） { display-mode: "form" }
# ※ ネットワーク環境によっては使えないことがあります。その場合は「ソロ練習」で開発を。
import os, re, subprocess, urllib.request

if not os.path.exists("cloudflared"):
    print("cloudflared をダウンロード中...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "cloudflared")
    os.chmod("cloudflared", 0o755)

proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:%d" % PORT],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for line in proc.stdout:                      # 出力から公開URLを拾う
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0); break
print("=" * 50)
print("公開URL:", public_url)
print("このURLを参加者に伝えてください。")
print("同じURLをブラウザで開くと、盤面（スクリーン用）が見られます。")
print("=" * 50)

## 盤面をスクリーンに映す

発行された **公開URL をブラウザで開く** と、リアルタイムの盤面が見られます。これをプロジェクタに映してください。このノート内でも下のセルで確認できます。

In [ ]:
# 【先生】盤面をこのノート内でも確認する（Colab専用。本番はブラウザで公開URLを開いて投影）
try:
    from google.colab.output import serve_kernel_port_as_iframe
    serve_kernel_port_as_iframe(PORT, path="/")
except Exception:
    print("Colab以外では、公開URL または http://127.0.0.1:%d/ をブラウザで開いてください" % PORT)

---
# B. 生徒セクション

各自のColabで、サーバに接続する **bot** を作ります。

## APIの仕様

| やりたいこと | 呼び出し | 返り値 |
|---|---|---|
| 参加する | `POST /join` `{"name": "名前"}` | `{"id": 自分の番号, "w":24, "h":24}` |
| 今の盤面を見る | `GET /state` | `{"coins": [[x,y]...], "players": [{"id","name","x","y","score","color"}...]}` |
| 動く | `POST /move` `{"id": 自分の番号, "dir": 向き}` | `{"x","y","score"}` |

- 向き `dir` は `"up"` / `"down"` / `"left"` / `"right"`
- 座標は **x=右方向、y=下方向**（左上が 0,0）

## 接続設定

先生に教わった **公開URL** を `SERVER` に貼り付け、API呼び出しヘルパー `api()` を用意します。

In [ ]:
import requests

# ★ 先生に教えてもらった公開URLをここに貼る（末尾のスラッシュは不要）
SERVER = "https://ここに先生のURL.trycloudflare.com"

def api(path, data=None):
    """サーバのAPIを呼ぶ。data を渡すと POST、渡さないと GET。返事は辞書。"""
    if data is None:
        return requests.get(SERVER + path).json()
    return requests.post(SERVER + path, json=data).json()

## 戦略を試す道具（`play`）

botの「頭脳」は、**盤面 `state` と自分 `me` を受け取り、動く向きを返す関数** `decide` です。
下の `play(decide, 名前, 秒数)` を使うと、その `decide` で一定時間プレイして **得点** を返します。これで各段階のbotを気軽に試せます。

In [ ]:
import time

def play(decide, name="あなた", seconds=15):
    """decide(state, me) が返す向きに従って seconds 秒プレイし、最終得点を返す。"""
    me = api("/join", {"name": name})
    my_id = me["id"]
    end = time.time() + seconds
    while time.time() < end:
        state = api("/state")
        me_now = next(p for p in state["players"] if p["id"] == my_id)
        api("/move", {"id": my_id, "dir": decide(state, me_now)})
        time.sleep(0.08)
    final = next(p for p in api("/state")["players"] if p["id"] == my_id)
    print("{} の得点: {}".format(name, final["score"]))
    return final["score"]

## まず接続確認（最小のbot）

参加して、ランダムに少し動くだけのbotです。盤面に自分の点が現れて動けば、接続成功です。

In [ ]:
# まず接続確認：参加して、ランダムに30歩だけ動く最小のbot
import time, random

me = api("/join", {"name": "テスト"})   # 参加すると自分の id がもらえる
my_id = me["id"]
print("参加できました。あなたの id =", my_id)

for _ in range(30):
    api("/move", {"id": my_id, "dir": random.choice(["up", "down", "left", "right"])})
    time.sleep(0.1)
print("ランダムに30歩 動きました（盤面で自分の点が動いたか確認しよう）")

---
# ソロ練習

先生の公開URLがまだ無くても、**自分のColabの中にアリーナと敵を用意**して、botを鍛えられます。

手順:
1. 「共通」の **board.html** と **アリーナサーバ** を実行
2. 「接続設定」と「play」のセルを実行
3. 下のセルを実行（ローカルのアリーナ＋ダミー敵＋盤面。接続先が自分のローカルに切り替わります）
4. 続く **L1〜L4** の `decide` を書いて `play()` で試す

In [ ]:
#@title ▶ ソロ練習の準備（サーバ＋ダミー敵） { display-mode: "form" }
# 先に「board.html」「アリーナサーバ」「接続設定」のセルを実行しておくこと
import threading, time, random

srv_local, PORT = start_server()
SERVER = "http://127.0.0.1:%d" % PORT   # 接続先を自分のローカルサーバに切り替え
print("ローカルのアリーナ:", SERVER)

def dummy(name):                         # ランダムに動くだけのダミー敵
    m = api("/join", {"name": name}); mid = m["id"]
    while True:
        api("/move", {"id": mid, "dir": random.choice(["up", "down", "left", "right"])})
        time.sleep(0.15)
for nm in ["敵A", "敵B", "敵C"]:
    threading.Thread(target=dummy, args=(nm,), daemon=True).start()
print("ダミーの敵を3体 動かしました。")

try:                                     # 盤面表示（Colab専用）
    from google.colab.output import serve_kernel_port_as_iframe
    serve_kernel_port_as_iframe(PORT, path="/")
except Exception:
    print("盤面は http://127.0.0.1:%d/ をブラウザで開いてください" % PORT)

print("この状態で、上の『自分のbotを作ろう』のセルを実行すると対戦できます。")

---
## 戦略を段階的に強くする（L1 → L4）

いきなり最強を目指さず、4段階で少しずつ強くします。各レベルの `decide` を書いて `play()` で試し、**ダミー敵より高得点**を目指しましょう。

### L1: 最寄りのコインへ向かう（型を見せる例）

まずは基本の型です。**一番近いコイン**の方向へ1歩ずつ進みます。これを動かして「decide は盤面 `state` と自分 `me` を受け取り、`"up"/"down"/"left"/"right"` を返す関数」という形をつかみましょう。次の L2 からは、これを参考に自分で書きます。

- 距離は `abs(コインx - 自分x) + abs(コインy - 自分y)`（マンハッタン距離）

In [ ]:
def decide(state, me):
    # 一番近いコインを選ぶ
    cx, cy = min(state["coins"], key=lambda c: abs(c[0]-me["x"]) + abs(c[1]-me["y"]))
    # その方向へ1歩
    if cx > me["x"]: return "right"
    if cx < me["x"]: return "left"
    if cy > me["y"]: return "down"
    return "up"

play(decide, "L1", seconds=15)   # ダミー敵より高得点なら成功

### L2: 他プレイヤーを避けて、横取りする

L1は他人を無視していました。今度は `state["players"]` で **相手の位置** も見て、
**自分の方が先に取れそうなコイン**を狙います（相手のすぐ近くのコインは避ける）。

下のセルに、decide を **自分で** 書きましょう（L1が参考になります。使うのは辞書・リスト・条件分岐・距離だけ）。

In [ ]:
# 「自分が相手より先に取れるコイン」を狙う decide を自分で書こう
#   返すのは "up"/"down"/"left"/"right"
#   state["coins"]   = [[x, y], ...]
#   state["players"] = [{"id","name","x","y","score","color"}, ...]（自分以外が相手）
#   距離は abs(x差) + abs(y差)（L1と同じ考え方）


play(decide, "L2", seconds=15)

### L3: 「評価点」で選ぶ（効率よく回る）

「一番近い」だけでなく、コインごとに **評価点** を計算して一番良いコインを選ぶと、工夫を足しやすくなります。
例：近いほど高評価、さらに **近くに別のコインが密集** しているほど加点（連続で取れる）。

In [ ]:
# コインごとに「評価点」をつけて、一番良いコインへ向かう decide を書こう
#   近いほど高評価。密集や端なども自由に加味してよい
#   ヒント: max(coins, key=評価する関数) が便利


play(decide, "L3", seconds=15)

### L4: 総仕上げ — あなただけの最強bot

L1〜L3の工夫を全部入れて、自分だけの `decide` を完成させましょう。これが本番で戦うbotになります。

In [ ]:
# L1〜L3の工夫を全部入れた、あなただけの最強 decide を書こう（これで本番に出る）


play(decide, "L4テスト", seconds=15)

---
## 本番：クラス対戦

完成した `decide`（L4）で、**先生の公開URL**につないで全員と競います。

1. `SERVER` を **先生のURL** に変更
2. 下のセルを実行（`seconds` は対戦時間に合わせて長めに）
3. 先生のスクリーンで自分の点の動きとランキングを確認

In [ ]:
# 本番：先生の公開URLに変更してから実行
SERVER = "https://ここに先生のURL.trycloudflare.com"

play(decide, "あなたの名前", seconds=180)   # L4 の decide で参加

---
# 不正（チート）とその対策

このアリーナは、実は **id さえ知っていれば他人のプレイヤーも動かせて** しまいます。通信は前回学んだとおり **HTTPで丸見え** なので、覗いて真似すれば「なりすまし移動」ができてしまうのです。

「不正ができてしまう」と分かるのは、**仕組みを理解できた証拠**。では、どう防ぐか——前回の **ハッシュ／デジタル署名** の出番です。

---
# 発展: 人間プレイヤーとして参加する（Gradio操作パネル）

これまでプレイヤーは **bot（プログラム）** でした。前回学んだ **Gradio** で **操作パネル** を作れば、**人間がボタンで参加**して、みんなのbotに混じって遊べます。

仕組みは数当ての公開と同じ「Gradio ← `requests` → アリーナサーバ」。ボタンを押すと `/move` をサーバへ送り、`/state` を取り直して盤面を絵にして表示します。

- 盤面を絵に描く `render()` は用意してあります（`PIL`）
- あなたが書くのは、ボタンが押されたときに **サーバへ動きを送る** 1行です

先に「接続設定」（`SERVER` と `api`）と、対戦相手（先生のサーバ / ソロ練習）を用意しておいてください。

In [ ]:
!pip install --quiet gradio

In [ ]:
import gradio as gr
from PIL import Image, ImageDraw

CELL = 16   # 1マスの大きさ（ピクセル）

def render(state):
    """盤面の状態を絵にする"""
    W, H = state["w"], state["h"]
    img = Image.new("RGB", (W * CELL, H * CELL), "#161b22")
    d = ImageDraw.Draw(img)
    for x, y in state["coins"]:                        # コイン（黄）
        d.ellipse([x*CELL+4, y*CELL+4, x*CELL+CELL-4, y*CELL+CELL-4], fill="#f1c40f")
    for p in state["players"]:                         # プレイヤー（各色）
        x, y = p["x"], p["y"]
        d.ellipse([x*CELL+2, y*CELL+2, x*CELL+CELL-2, y*CELL+CELL-2], fill=p["color"])
    return img

my_id = None

def join_game(name):
    global my_id
    my_id = api("/join", {"name": name})["id"]         # 参加して自分のidをもらう
    return render(api("/state")), "参加しました（id={}）".format(my_id)

def move(direction):
    if my_id is None:
        return None, "先に「参加する」を押してください"

    # ここで、direction 方向へ /move する（接続設定の api を使う）


    state = api("/state")                              # 最新の盤面を取り直す
    me = next((p for p in state["players"] if p["id"] == my_id), None)
    return render(state), "あなたのスコア: {}".format(me["score"] if me else 0)

# --- 操作パネルの見た目 ---
with gr.Blocks() as demo:
    gr.Markdown("## アリーナ操作パネル（人間プレイヤー）")
    name_in = gr.Textbox(label="名前", value="人間")
    join_btn = gr.Button("参加する", variant="primary")
    board = gr.Image(label="盤面")
    info = gr.Textbox(label="状態")
    up = gr.Button("↑ 上")
    with gr.Row():
        left = gr.Button("← 左"); down = gr.Button("↓ 下"); right = gr.Button("→ 右")

    join_btn.click(join_game, inputs=name_in, outputs=[board, info])
    up.click(lambda: move("up"),       outputs=[board, info])
    down.click(lambda: move("down"),   outputs=[board, info])
    left.click(lambda: move("left"),   outputs=[board, info])
    right.click(lambda: move("right"), outputs=[board, info])

demo.launch(share=True)

In [ ]:
# 実験：このサーバは id さえ知っていれば「他人のプレイヤーも動かせて」しまう（認証が無いため）
try:
    print(api("/move", {"id": 0, "dir": "up"}))   # 他人(id=0)でも動いてしまうかも
except Exception as e:
    print("エラー:", e)

# 対策の考え方（前回の「ハッシュ／署名」の応用）:
#  1) join のときサーバが各プレイヤーに「秘密トークン」を配る
#  2) move には (id, dir, トークンから作った署名) を付けて送る
#  3) サーバは署名を検証し、正しい持ち主だけ動かす
# → 通信が丸見えでも、署名は本人しか作れないので「なりすまし移動」を防げる。

---
# おわりに

ここまでで、あなたは自分の手で **サーバを立て、通信し、botを動かし、全員で競い、そして不正と対策まで** 体験しました。

- **クラス**でデータの設計図を作り
- **バイナリとセキュリティ**でデータの正体と守り方を知り
- **ネットワーク**で通信の仕組みを作り
- そして今日、それらを **全部つないで** 動くものを作りました

ここからは、ルールを変える・戦略を磨く・不正対策を実装する——好きに拡張してみてください。おつかれさまでした。